In [3]:
# Part D — Data Integration Strategy

# This notebook outlines a conceptual strategy for integrating the selected TCGA-OV radiology and proteogenomics datasets. 


In [5]:
# Datasets to be Integrated

#1. TCGA-OV Radiology (TCIA) -- -- Imaging metadata covering 143 patients and 844 imaging series.

#2. TCGA-OV Proteogenomics (TCIA) -- -- Imaging metadata for a subset of 20 TCGA-OV patients with associated proteogenomic profiling.

# Both datasets use a shared patient identifier system based on TCGA subject IDs, represented in the metadata as the `Patient ID` field. This enables direct linkage between the radiology and proteogenomics datasets at the patient level.

# Integration would be performed using a patient-level join on `Patient ID`, with the proteogenomics cohort acting as a subset of the full radiology population.


In [13]:
import pandas as pd

In [14]:
radiology = pd.read_excel("../data/raw/tcga-ov/TCIA_TCGA-OV_09-16-2015-nbia-digest.xlsx")
proteogenomics = pd.read_excel("../data/raw/tcga_ov_proteogenomics/2020-02-26_Images_from_HGSOC_proteomics-nbia-digest.xlsx")

In [15]:
radiology_patient = (radiology.groupby("Patient ID").agg(
        n_studies=("Study Instance UID", "nunique"),
        n_series=("Series Instance UID", "nunique"),
        total_images=("Image Count", "sum"),
        modality=("Modality", lambda x: x.mode().iloc[0]),
        manufacturer=("Manufacturer", lambda x: x.mode().iloc[0])).reset_index())

radiology_patient = radiology_patient.rename(columns={"Patient ID": "patient_id_canonical"})

In [16]:
proteogenomics_patient = (proteogenomics.groupby("Patient ID").agg(
        pg_n_studies=("Study Instance UID", "nunique"),
        pg_n_series=("Series Instance UID", "nunique"),
        pg_total_images=("Image Count", "sum"),
        pg_manufacturer=("Manufacturer", lambda x: x.mode().iloc[0])).reset_index())

proteogenomics_patient = proteogenomics_patient.rename(columns={"Patient ID": "patient_id_canonical"})

In [17]:
ovarian_universal = radiology_patient.merge(proteogenomics_patient, on="patient_id_canonical", how="left")
ovarian_universal.to_csv("../data/processed/ovarian_universal.csv", index=False)
total_patients = ovarian_universal.shape[0]
linked_patients = ovarian_universal["pg_n_studies"].notna().sum()
match_rate = linked_patients / total_patients
total_patients, linked_patients, match_rate

(143, 20, 0.13986013986013987)